# **Demo 03: Building a Reactive Agent Workflow with LangGraph and Tavily**

**Objective:** Build a reactive AI agent using LangGraph integrated with LangChain's Tavily search tool. This demo walks through creating a react-style agent via `create_agent()`, connecting it to a real-time web search tool, and invoking the agent to respond dynamically to user queries. It showcases how LangGraph enables modular, tool-enhanced agent workflows for tasks requiring external information retrieval and context-aware decision-making.

**Prerequisites:**
- Tavily API key (for web search)
- At least one LLM provider key: Groq, HuggingFace, Azure OpenAI, or OpenAI

**Tools &amp; Libraries:**
- Python 3.10+
- `langgraph` — Agent orchestration framework
- `langchain-openai` — OpenAI-compatible LLM interface
- `langchain-tavily` — Tavily web search integration
- `python-dotenv` — Environment variable management

**Scenario:** : A product lead at a fintech startup wants to monitor global developments in real-time fraud detection methods for UPI systems. The manual process of tracking cybersecurity advancements and regulatory changes is inefficient and prone to oversight. The goal is to automate this workflow using LangGraph, where an agent uses a search tool to gather the latest research and another agent validates and summarizes the findings for internal teams.

In [ ]:
# Step 1: Install Required Libraries
# Installs LangGraph (agent framework), LangChain OpenAI adapter, Tavily search
# integration, and python-dotenv for loading API keys from a .env file.
%pip install langgraph langchain-openai langchain-tavily tavily-python python-dotenv

In [ ]:
# Step 2: Load Environment Variables from .env File
# Reads key-value pairs from the .env file and sets them as environment variables.
# This avoids issues with load_dotenv() path resolution in notebook environments.
# The .env file should contain: TAVILY_API_KEY, and at least one LLM provider key.

import os

_env_file = r"c:\Nikhil\Work_Related\AI\MSAGI\AI\5-Multi-Agent Ecosystem 1\Demos_updated\Lesson_02\Demo_3\.env"
with open(_env_file, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()

# Verify that each key is present (prints True/False — never prints the actual key values)
print("TAVILY_API_KEY loaded:", bool(os.environ.get("TAVILY_API_KEY")))
print("GROQ_API_KEY loaded:", bool(os.environ.get("GROQ_API_KEY")))
print("GEMINI_API_KEY loaded:", bool(os.environ.get("GEMINI_API_KEY")))
print("HF_API_KEY loaded:", bool(os.environ.get("HF_API_KEY")))
print("DEEPSEEK_API_KEY loaded:", bool(os.environ.get("DEEPSEEK_API_KEY")))

In [ ]:
# Step 3: Configure the LLM (Large Language Model)
# ==============================================================================
# This demo supports five interchangeable LLM providers. Each exposes an
# OpenAI-compatible chat-completions endpoint, allowing LangGraph to use them
# via ChatOpenAI with no code changes — just swap the LLM_PROVIDER value.
#
# FREE providers (recommended):
#   "groq"        — Llama 3.3 70B on Groq (very fast, free tier, tool calling ✓)
#                   Best free option: generous rate limits, handles multi-step agents well.
#   "gemini"      — Google Gemini 2.0 Flash (free but only 15 RPM — can hit rate limits
#                   with ReAct agents that make multiple LLM calls per invocation)
#   "huggingface" — Llama 3.3 70B via HuggingFace Inference Router (free, tool calling ✓)
#
# PAID providers:
#   "deepseek"    — DeepSeek-Chat via DeepSeek API (tool calling ✓, requires balance)
#   "azure"       — Azure OpenAI / GPT models (tool calling ✓)
#
# NOTE: The react agent requires a model that supports tool/function calling.
# ==============================================================================
from langchain_openai import ChatOpenAI, AzureChatOpenAI

LLM_PROVIDER = "groq"  # <-- Change this to switch providers

if LLM_PROVIDER == "huggingface":
    # Llama 3.3 70B via HuggingFace's free Inference Router — strong reasoning & tool calling
    hf_key = os.getenv("HF_API_KEY")
    if not hf_key:
        raise ValueError("HF_API_KEY not found in environment variables.")
    llm = ChatOpenAI(
        model="meta-llama/Llama-3.3-70B-Instruct",
        api_key=hf_key,
        base_url="https://router.huggingface.co/v1",
        temperature=0.5,
    )
elif LLM_PROVIDER == "groq":
    # Llama 3.3 70B on Groq — fastest free option with full tool calling support
    groq_key = os.getenv("GROQ_API_KEY")
    if not groq_key:
        raise ValueError("GROQ_API_KEY not found in environment variables. Add it to your .env file.")
    llm = ChatOpenAI(
        model="llama-3.3-70b-versatile",
        api_key=groq_key,
        base_url="https://api.groq.com/openai/v1",
        temperature=0.5,
    )
elif LLM_PROVIDER == "gemini":
    # Google Gemini 2.0 Flash — free tier (15 requests/min, 1M tokens/day)
    # WARNING: ReAct agents make multiple LLM calls per invoke, which can easily
    # exhaust the 15 RPM free limit. Use "groq" for more reliable free usage.
    gemini_key = os.getenv("GEMINI_API_KEY")
    if not gemini_key:
        raise ValueError("GEMINI_API_KEY not found. Get a free key at https://aistudio.google.com/apikey")
    llm = ChatOpenAI(
        model="gemini-2.0-flash",
        api_key=gemini_key,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        temperature=0.5,
    )
elif LLM_PROVIDER == "azure":
    llm = AzureChatOpenAI(
        azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        temperature=0.5,
    )
elif LLM_PROVIDER == "deepseek":
    # DeepSeek official API — requires paid balance at https://platform.deepseek.com
    ds_key = os.getenv("DEEPSEEK_API_KEY")
    if not ds_key:
        raise ValueError("DEEPSEEK_API_KEY not found. Get one at https://platform.deepseek.com")
    llm = ChatOpenAI(
        model="deepseek-chat",
        api_key=ds_key,
        base_url="https://api.deepseek.com",
        temperature=0.5,
    )
else:
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}. Use 'groq', 'gemini', 'huggingface', 'deepseek', or 'azure'.")

print(f"LLM configured: {LLM_PROVIDER}")

In [ ]:
# Step 4: Define the Tavily Web Search Tool
# Creates a LangChain tool that wraps TavilySearch. When the agent decides it needs
# external information, it calls this tool with a search query. Tavily returns the
# top 3 web results, which the agent can then use to formulate its response.

from langchain_tavily import TavilySearch
from langchain.tools import tool

@tool #python decorator (want this function to behave like a tool that the agent can call)
def search_tool(query: str):
    # This is a doc-string that describes the tool's purpose. It will be visible to the agent when it considers which tool to call.
    # This helps for the documentation of the function, when we hover it will explain what the function does.
    """Use this tool to search the web for real-time information via the Tavily API."""
    tavily_search_tool = TavilySearch(max_results=3)
    result = tavily_search_tool.invoke(query)
    return result

In [ ]:
# Step 5: Create the Reactive Agent Using LangGraph
# Uses LangGraph's create_agent() to build a ReAct-style agent that combines the LLM
# with the Tavily search tool. The agent follows a Reason → Act → Observe loop:
#   1. The LLM reasons about the user query
#   2. If external info is needed, it calls the search tool (Act)
#   3. It observes the tool output and formulates a final answer

from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[search_tool])

In [ ]:
# Step 6: Visualize the Agent Graph
# Displaying the agent object renders its internal state graph, showing the nodes
# (model, tools) and edges (decision paths) that make up the ReAct workflow.
agent

In [ ]:
# Step 7: Invoke the Agent WITH the Search Tool
# Sends a domain-specific query that requires real-time web data. The agent will
# recognize it needs external information, call the Tavily search tool, and
# synthesize the results into a coherent answer.

state = agent.invoke({
    "messages": [
        {"role": "user", "content": "How to be a good Principal Enterprise Architect at an Enterprise and what are the steps to achieve it? "
        "What would the path look like to transition from a Principal Enterprise Architect to a Chief Enterprise Architect to a CTO or CIO?"}
    ]
})

print("\n[With Tavily Search Tool]:\n", state)

In [ ]:
# Step 8: Invoke the Agent WITHOUT the Search Tool (Pure LLM Reasoning)
# Sends a simple math question that the LLM can answer directly from its training
# data. The agent recognizes no tool call is needed and responds immediately even though it has access to the search tool.,
# demonstrating the ReAct agent's ability to skip tools when unnecessary.

state = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is 2 + 3?"}
    ]
})

print("\n[Without Tool — Pure LLM Reasoning]:\n", state)

#### Summary
By following these steps you have:
1. Installed and configured all required libraries
2. Loaded API keys securely from a `.env` file
3. Set up a configurable LLM provider (Groq / HuggingFace / Azure / DeepSeek)
4. Defined a Tavily web search tool for real-time information retrieval
5. Built a **ReAct-style agent** using LangGraph's `create_agent()`
6. Visualized the agent's internal state graph
7. Demonstrated the agent **using a tool** (web search for fraud detection methods)
8. Demonstrated the agent **without a tool** (direct LLM reasoning for simple math)

This showcases LangGraph's modular, tool-enhanced agent workflows for external retrieval and context-aware decision-making.

In [ ]:
# Example of how to set up a node and edge along with guards.
graph = agent.get_graph()
graph.render("agent_workflow", format="png", cleanup=True)
graph.add_node ("node1", agent 1)
graph.add_node("node2", agent 2)
graph.add_edge("node1", "node2")